In [2]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy
from pydantic import BaseModel, Field 
from typing import Literal



In [2]:
load_dotenv()

True

In [4]:
model = ChatOpenAI(model="gpt-4o-mini")

In [8]:
prompt = """
Understand the coding request:

For any request, identify the following :

- task type
- objective
- files involved 

Return the response in the following JSON format:
{
    "task_type": "type of task",
    "objective": "objective of the task",
    "files_involved": "list of files involved"
}

"""

In [5]:
request = "indentify the bug in the auth.py file where the exchange token is not getting generated"

In [6]:
response = model.invoke(prompt+request)
print(response.content)

```json
{
    "task_type": "bug identification",
    "objective": "identify the bug in the auth.py file where the exchange token is not getting generated",
    "files_involved": ["auth.py"]
}
```


In [6]:
class CodingRequest(BaseModel):
    """Structured representation of the coding request"""

    task_type: Literal[
        "implement",
        "debug",
        "review",
        "explain"
    ] = Field(description="Type of task to be performed")

    objective: str = Field(description="A concise description of what needs to be achieved")

    need_code_changes: bool = Field(
        description="Whether fulfilling this task required modifying the source code"
    )

    target_files: list[str] = Field(
        description="List of files that are relevant to the task, e.g. ['auth.py', 'models.py']"
    )






In [12]:
structured_output_model = model.with_structured_output(CodingRequest)

In [15]:
result = structured_output_model.invoke("""
       Fix the login bug in auth.py. Expired session currently produce HTTP 500.
""")

In [17]:
print(type(result))

<class '__main__.CodingRequest'>


In [18]:
CodingRequest.model_json_schema()

{'description': 'Structured representation of the coding request',
 'properties': {'task_type': {'description': 'Type of task to be performed',
   'enum': ['implement', 'debug', 'review', 'explain'],
   'title': 'Task Type',
   'type': 'string'},
  'objective': {'description': 'A concise description of what needs to be achieved',
   'title': 'Objective',
   'type': 'string'},
  'need_code_changes': {'description': 'Whether fulfilling this task required modifying the source code',
   'title': 'Need Code Changes',
   'type': 'boolean'},
  'target_files': {'description': "List of files that are relevant to the task, e.g. ['auth.py', 'models.py']",
   'items': {'type': 'string'},
   'title': 'Target Files',
   'type': 'array'}},
 'required': ['task_type', 'objective', 'need_code_changes', 'target_files'],
 'title': 'CodingRequest',
 'type': 'object'}

In [23]:
agent = create_agent(
    model= model,
    tools = [],
    response_format=ProviderStrategy(CodingRequest),
    system_prompt=prompt,
)


In [24]:
result = agent.invoke(
    {
    "messages" : [
         {"role": "user", "content":"Indentify the bug in the auth.py file where the exchange token is not getting generated"}
    ]
    }
)
print(type(result["structured_response"]))
print(result["structured_response"])

<class '__main__.CodingRequest'>
task_type='debug' objective='Identify the bug in the auth.py file where the exchange token is not getting generated' need_code_changes=False target_files=['auth.py']


In [25]:
for message in result["messages"]:
    print(message.pretty_print())
    print("============\n")

================================ Human Message =================================

Indentify the bug in the auth.py file where the exchange token is not getting generated
None

================================== Ai Message ==================================

{"task_type":"debug","objective":"Identify the bug in the auth.py file where the exchange token is not getting generated","need_code_changes":false,"target_files":["auth.py"]}
None

